In [1]:
import os
import stat

if os.environ.get("GDMSESSION", None) is not None:
    runtime_dir = '/root/capsule/scratch/runtime-root'
    if not os.path.exists(runtime_dir):
        os.makedirs(runtime_dir)
    os.chmod(runtime_dir, stat.S_IRWXU)
    os.environ['XDG_RUNTIME_DIR'] = runtime_dir
    %gui qt

Qt: Session management error: None of the authentication protocols specified are supported


In [2]:
from pathlib import Path
import spatialdata as sd
import pandas as pd
import numpy as np
import gc

from xenium_analysis_tools.alignment.format_for_napari import (
    add_mapped_cells_cols,
    get_plot_sdata,
)

data_root = Path('/root/capsule/data')
data_folder = data_root / "xenium_756772_napari"

In [3]:
combined_data_path = data_folder / "combined_data.zarr"
combined_data = sd.read_zarr(combined_data_path)

no parent found for <ome_zarr.reader.Label object at 0x7fc6484076d0>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc648421590>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc648432c10>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc6482622d0>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc648422450>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc648430d10>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc6482da010>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc6482c7ad0>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc6484313d0>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc6482f15d0>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc648253750>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc6482f87d0>: None
no parent found for <ome_zarr.reader.Label object at 0x7fc6482e7690>: None
no parent found for <ome_

## Add additional columns

In [4]:
mapping_output_path = data_folder / 'xenium_756772_mapping/mapped_data/mapped_cellxgene.h5ad'
add_scanpy_metrics = True

# Mapped types
if mapping_output_path is not None:
    import anndata as ad
    from xenium_analysis_tools.map_xenium.format_mapping import (
        add_broad_types,
        get_shared_colormap,
        add_colormap_adata
    )
    mapped_data = ad.read_h5ad(mapping_output_path)
    mapped_data.obs['cell_id'] = mapped_data.obs['original_cell_id']
    mapped_data.obs.rename_axis('cell_section_id', inplace=True)
    mapped_data.var.set_index('gene_symbol', inplace=True, drop=False)
    combined_data['table'] = add_mapped_cells_cols(combined_data['table'], mapped_data, verbose=True)
    combined_data['table'] = add_broad_types(combined_data['table'])
    shared_cmp = get_shared_colormap(combined_data['table'])
    combined_data['table'] = add_colormap_adata(combined_data['table'], shared_cmp)


# Scanpy metrics
if add_scanpy_metrics:
    import scanpy as sc
    combined_data['table'].var['is_gene_expression'] = combined_data['table'].var['feature_types'] == 'Gene Expression'
    sc.pp.calculate_qc_metrics(combined_data['table'], qc_vars=['is_gene_expression'], inplace=True)
    combined_data['table'].obs['n_genes_expressed'] = (combined_data['table'][:, combined_data['table'].var['is_gene_expression']].X > 0).sum(axis=1)
    print(f"\n{combined_data['table'].obs[['n_genes_by_counts', 'n_genes_expressed', 'total_counts_is_gene_expression']].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])}\n")

Adding 51 obs columns: ['class_label', 'class_name', 'class_bootstrapping_probability', 'class_aggregate_probability', 'class_avg_correlation', 'class_directly_assigned', 'subclass_label', 'subclass_name', 'subclass_bootstrapping_probability', 'subclass_aggregate_probability', 'subclass_avg_correlation', 'subclass_directly_assigned', 'supertype_label', 'supertype_name', 'supertype_bootstrapping_probability', 'supertype_aggregate_probability', 'supertype_avg_correlation', 'supertype_runner_up_assignment_0', 'supertype_runner_up_assignment_1', 'supertype_runner_up_correlation_0', 'supertype_runner_up_correlation_1', 'supertype_runner_up_probability_0', 'supertype_runner_up_probability_1', 'supertype_directly_assigned', 'cluster_label', 'cluster_name', 'cluster_alias', 'cluster_bootstrapping_probability', 'cluster_aggregate_probability', 'cluster_avg_correlation', 'cluster_runner_up_assignment_0', 'cluster_runner_up_correlation_0', 'cluster_runner_up_probability_0', 'cluster_directly_assi

# FOV detail view with GCaMP, RNA, cell labels, and transcripts

In [6]:
# =============================================================================
# PLOT 1: FOV detail ? GCaMP, RNA channel, cell labels, and transcripts
# =============================================================================
# This plot shows a single section aligned into the GCaMP FOV. You can explore:
#   - GCaMP signal and cell labels (gcamp_labels)
#   - The Xenium RNA channel aligned to the z-stack
#   - Cell segmentation labels (cell_labels)
#   - Transcripts: all genes, or a filtered subset
#
# DATA NEEDED: fov_data with section-specific elements
# COORDINATE SYSTEM: czstack_microns
# =============================================================================

import time
import warnings
import numpy as np
from xenium_analysis_tools.alignment.format_for_napari import filter_cells, filter_transcripts, drop_2ds, get_fov_sdata, filter_labels

# ---- Set up --------------------------------------------------------------
coord_sys = 'czstack_microns'
sections_to_view = [3]     

# ---- Transcripts  -------------------------------------------------
genes_to_show = ['EGFP', 'Gad2', 'Sst', 'Pvalb', 'Vip']                 # Set to 'all' to show every gene, or provide a list to filter to specific genes
feature_colormap = {
    'EGFP': '#ff00c8',  
    'Gad2': '#ff7f0e',  
    'Sst':  '#990606',  
    'Pvalb':'#9102f7',  
    'Vip':  '#ffff0f'   
}
min_qv = 20                                                             # Minimum quality value threshold (>20 qv included as 'count')
filter_is_gene = True                                                   # Filter to only "is_gene" transcripts (excludes blanks/controls)  
filter_assigned_to_cell = True                                          # Filter to only transcripts assigned to a cell
filter_transcripts_to_cells = True                                      # Filter to transcripts in cells that pass filters

# ---- Cells --------------------------------------------------------
cell_filters = [
    {'column': 'n_genes_expressed',  'operator': '>=',    'value': 5},
    {'column': 'transcript_counts',  'operator': '>=',    'value': 10}
]

# --------------------------------------------------------------------------

# Contrast limits per channel
channel_contrast = {
    'gcamp':       (0, 12000),
    'rna':         (0, 12000),
    'protein':     (0, 12000),
    'boundary':    (0, 11000),
    'dapi_zstack': (0, 12000),
    'dapi':        (0, 12000),
}

plot_3D = True

# -----------------------------------------------------------------------------
# Which morphology focus channels to drop from the plot (set to [] to keep all)
drop_mf_chans = ['protein', 'boundary', 'dapi']

# Build the filtered data subset for this section
plot_sdata_fov = get_plot_sdata(
    combined_data,
    sections_to_plot=sections_to_view,
    elements_to_plot=['gcamp', 'morphology_focus', 'dapi_zstack', 'cell_labels', 'gcamp_labels', 'transcripts', 'table', 'gcamp_table'],
    include_zstack=True,
    split_morphology=True,  
    lift_to_3d=True,        
    lift_n_z=combined_data['dapi_zstack-1'].shape[1],
)

# Drop morphology focus channels we don't want
for ch in drop_mf_chans:
    els = [el for el in plot_sdata_fov.images if el.startswith(ch) and not el.startswith('dapi_zstack')]
    for el in els:
        print(f"Dropping channel '{ch}' element '{el}' from plot.")
        del plot_sdata_fov.images[el]

# Drop 2D elements
plot_sdata_fov = drop_2ds(plot_sdata_fov)

# Crop to FOV
plot_sdata_fov = get_fov_sdata(plot_sdata_fov)

# ---- Apply cell/table filters --------------------------------------------
plot_sdata_fov = filter_cells(plot_sdata_fov, cell_filters)

# ---- Filter labels to included cells -------------------------------------
plot_sdata_fov = filter_labels(plot_sdata_fov)

# ---- Apply transcript filters --------------------------------------------
plot_sdata_fov = filter_transcripts(plot_sdata_fov, 
                                    genes_to_show=genes_to_show,
                                    min_qv=min_qv,
                                    filter_is_gene=filter_is_gene,
                                    filter_assigned_to_cell=filter_assigned_to_cell,
                                    filter_transcripts_to_cells = filter_transcripts_to_cells
                                    )


# ---- Launch Napari -------------------------------------------------------------------------
import napari
from qtpy.QtWidgets import QApplication
from napari_spatialdata import Interactive
from napari_spatialdata.constants import config
warnings.filterwarnings("ignore", message="Non-orthogonal slicing is being requested")

# ---- Raise point threshold so all transcripts render ------------------------
tx_els = list(plot_sdata_fov.points.keys())
if tx_els:
    num_txs = [plot_sdata_fov[el].compute().shape[0] for el in tx_els]
    config.POINT_THRESHOLD = int(np.max([config.POINT_THRESHOLD, np.max(num_txs)]))

config.PROJECT_3D_POINTS_TO_2D = False
config.PROJECT_2_5D_SHAPES_TO_2D = False

# ---- Initialize viewer ------------------------------------------------------
interactive_fov = Interactive(plot_sdata_fov)
viewer_fov = interactive_fov._viewer
if plot_3D:
    viewer_fov.dims.ndisplay = 3

# ---- Define per-layer styles ------------------------------------------------
layer_styles = {
    'gcamp':       {'colormap': 'green',        'contrast_limits': channel_contrast['gcamp'],        'blending': 'additive', 'opacity': 1.0},
    'rna':         {'colormap': 'purple',       'contrast_limits': channel_contrast['rna'],          'blending': 'additive', 'opacity': 0.8},
    'protein':     {'colormap': 'magenta',      'contrast_limits': channel_contrast['protein'],      'blending': 'additive', 'opacity': 0.8},
    'boundary':    {'colormap': 'gray',         'contrast_limits': channel_contrast['boundary'],     'blending': 'additive', 'opacity': 0.8},
    'dapi':        {'colormap': 'blue',         'contrast_limits': channel_contrast['dapi'],         'blending': 'additive', 'opacity': 0.8},
    'dapi_zstack': {'colormap': 'bop blue',     'contrast_limits': channel_contrast['dapi_zstack'],  'blending': 'additive', 'opacity': 0.8},
    'landmarks' :  {'size': 10,                 'opacity': 1.0,   'blending': 'translucent', 'face_color': 'white', 'out_of_slice_display': True, 'symbol': 'star'},
    'gcamp_labels':{'contour': 2,               'opacity': 0.5,   'blending': 'translucent'},
    'cell_labels': {'contour': 2,               'opacity': 0.4,   'blending': 'translucent'},
    'transcripts': {'size': 1, 
                    'opacity': 0.8, 
                    'blending': 'translucent', 
                    'out_of_slice_display': True,
                    'face_color': 'feature_name',  # categorical string column
                    },
}

def apply_layer_style(layer, sdata):
    """Apply styles from layer_styles dict by matching on layer name prefix."""
    for key, params in layer_styles.items():
        if key in layer.name:
            if 'colormap' in params and isinstance(layer, napari.layers.Image):
                layer.colormap = params['colormap']
            if 'contrast_limits' in params and hasattr(layer, 'contrast_limits'):
                layer.contrast_limits = params['contrast_limits']
            if 'blending' in params and hasattr(layer, 'blending'):
                layer.blending = params['blending']
            if 'opacity' in params and hasattr(layer, 'opacity'):
                layer.opacity = params['opacity']
            if 'size' in params and hasattr(layer, 'size'):
                layer.size = params['size']
            if 'contour' in params and isinstance(layer, napari.layers.Labels):
                layer.contour = params['contour']
            if 'out_of_slice_display' in params and hasattr(layer, 'out_of_slice_display'):
                layer.out_of_slice_display = params['out_of_slice_display']
            if 'symbol' in params and hasattr(layer, 'symbol'):
                layer.symbol = params['symbol']

            print(f"  Styled '{layer.name}' as '{key}'")
            break

# ---- Load elements in order: images -> labels -> points ---------------------
els_fov = (
    list(plot_sdata_fov.images.keys()) +
    list(plot_sdata_fov.labels.keys()) +
    list(plot_sdata_fov.points.keys())
)

for el in els_fov:
    if el in viewer_fov.layers:
        print(f"Layer '{el}' already exists. Skipping.")
        continue

    expected_layers_count = len(viewer_fov.layers) + 1
    interactive_fov.add_element(el, element_coordinate_system=coord_sys)

    while len(viewer_fov.layers) < expected_layers_count:
        QApplication.processEvents()
        time.sleep(0.1)

    # Pass the spatial data object to apply_layer_style
    apply_layer_style(viewer_fov.layers[-1], plot_sdata_fov)
    print(f"Added: {el}")

# Move gcamp to the top
gcamp_layer = viewer_fov.layers['gcamp']
viewer_fov.layers.move(viewer_fov.layers.index(gcamp_layer), -1)

Dropping channel 'protein' element 'protein-3' from plot.
Dropping channel 'boundary' element 'boundary-3' from plot.
Dropping channel 'dapi' element 'dapi-3' from plot.
Removing 2D element: morphology_focus-3
Filter 'n_genes_expressed >= 5': 309 / 344 cells pass
Filter 'transcript_counts >= 10': 268 / 344 cells pass

268 / 344 cells kept after all filters.
transcripts-3: filters applied (lazy — will compute on render)


2026-03-30 06:52:49.732 | WARNING  | napari_spatialdata._viewer:__init__:56 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.
2026-03-30 06:52:51.484 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 06:52:51.485 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.


  Styled 'dapi_zstack-3' as 'dapi'
Added: dapi_zstack-3


2026-03-30 06:52:54.257 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 06:52:54.264 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 06:52:54.265 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.


  Styled 'gcamp' as 'gcamp'
Added: gcamp


...

  Styled 'cell_labels-3' as 'cell_labels'
Added: cell_labels-3


2026-03-30 06:53:07.394 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 06:53:07.404 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.


  Styled 'transcripts-3' as 'transcripts'
Added: transcripts-3


True

/opt/conda/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


# Plot full GCaMP z-stack with all boundary sections laid out across z

In [ ]:
# =============================================================================
# PLOT 2: GCaMP z-stack + all boundary sections laid out across cortical z-axis
# =============================================================================
# This plot shows the full spatial layout of all Xenium sections registered
# into the GCaMP z-stack coordinate system. Each section boundary is shown 
# in a distinct color so you can see where each section sits along z.
#
# COORDINATE SYSTEM: czstack_microns
# =============================================================================

# ---- Set up -----------------------------------------------------------------
coord_sys = 'czstack_microns'

sections_colormap = 'tab20'
gcamp_contrast_limits = (0, 12000)   
boundary_contrast_limits = (0, 11000)
boundary_opacity = 0.75
# -----------------------------------------------------------------------------

# Build the data subset - only gcamp + boundaries 
plot_sdata_overview = get_plot_sdata(
    combined_data,
    sections_to_plot='all',                         
    elements_to_plot=['gcamp', 'morphology_focus'],  
    include_zstack=True,
    split_morphology=True,                           
    lift_to_3d=True,                                 
    lift_n_z=combined_data['dapi_zstack-1'].shape[1],
)

# ---- Launch Napari -------------------------------------------------------------------------
import time
import napari
import warnings
import logging
import matplotlib.pyplot as plt
from qtpy.QtWidgets import QApplication
from napari_spatialdata import Interactive
from napari_spatialdata.constants import config

# Suppress known non-critical warnings
warnings.filterwarnings("ignore", message="Non-orthogonal slicing is being requested")
warnings.filterwarnings("ignore", message="data shape.*exceeds GL_MAX_TEXTURE_SIZE")
logging.getLogger("napari_spatialdata").setLevel(logging.WARNING)

# Initialize viewer
config.PROJECT_3D_POINTS_TO_2D = False
interactive_overview = Interactive(plot_sdata_overview)
viewer_overview = interactive_overview._viewer
viewer_overview.dims.ndisplay = 3

# Sort boundary elements numerically by section number
boundary_imgs = sorted(
    [el for el in plot_sdata_overview.images if el.startswith('boundary')],
    key=lambda x: int(x.rsplit('-', 1)[-1])
)
els = ['gcamp'] + boundary_imgs
cmap = plt.get_cmap(sections_colormap, len(boundary_imgs))
sections_colors_rgba = [cmap(i) for i in range(len(boundary_imgs))]
sections_colors_hex = [
    '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))
    for r, g, b, _ in sections_colors_rgba
]

for i, el in enumerate(els):
    if el in viewer_overview.layers:
        print(f"Layer '{el}' already exists. Skipping.")
        continue

    expected_layers_count = len(viewer_overview.layers) + 1
    interactive_overview.add_element(el, element_coordinate_system=coord_sys)

    while len(viewer_overview.layers) < expected_layers_count:
        QApplication.processEvents()
        time.sleep(0.1)

    new_layer = viewer_overview.layers[-1]
    new_layer.blending = 'translucent'
    new_layer.opacity = boundary_opacity

    if el == 'gcamp':
        new_layer.colormap = 'green'
        new_layer.contrast_limits = gcamp_contrast_limits
        new_layer.blending = 'translucent'
        new_layer.opacity = 1.0
    else:
        # Assign a distinct color per section (cycle if more sections than colors)
        sec_idx = i - 1  # offset for gcamp at index 0
        new_layer.colormap = napari.utils.Colormap(
            colors=['black', sections_colors_hex[sec_idx % len(sections_colors_hex)]], 
            name=f'section_{sec_idx}'
        )
        new_layer.contrast_limits = boundary_contrast_limits

    print(f"Added layer: {el}")

# Move gcamp to the top
gcamp_layer = viewer_overview.layers['gcamp']
viewer_overview.layers.move(viewer_overview.layers.index(gcamp_layer), -1)